In [4]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    # ソースコードを記述
    tool = TavilySearchResults(max_results=2)
    tools = [tool]

    # グラフのインスタンスを作成
    graph_builder = StateGraph(State)

    # 言語モデルの定義
    llm = ChatOpenAI(model_name=model_name)

    # 変更点：ツール定義の紐づけ
    llm_with_tools = llm.bind_tools(tools)

    # チャットボットノード作成
    def chatbot_node(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}
    
    graph_builder.add_node("chatbot", chatbot_node)

    # ツールノードの作成
    tool_node = ToolNode(tools)

    # グラフにツールノードを追加
    graph_builder.add_node("tools", tool_node)

    # 条件付エッジの作成
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition, # ツール呼出と判断したらツールノードを呼ぶ
    )

    # ツールが呼び出されるたびに、チャットボットに戻って次のステップを決定
    # ツールからチャットボットへの戻りエッジを作成
    graph_builder.add_edge("tools", "chatbot")

    # 開始ノードの指定
    graph_builder.set_entry_point("chatbot")

    # 記憶を持つ実行可能なステートグラフの作成
    memory = MemorySaver()
    return graph_builder.compile(checkpointer=memory)


# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    # ソースコードを記述
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values")
    # 結果をストリーミングで得る
    for event in events:
        print(event["messages"][-1].content, flush=True)


# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
# ソースコードを記述
graph = build_graph(MODEL_NAME)

# メインループ
# ソースコードを記述
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

こんにちは
こんにちは！今日はどんなことをお手伝いできますか？
ぽこあポケモンについて教えて

[{"url": "https://www.pocoapokemon.jp/ja/", "content": "いつかは  \n大きな街を作り、  \nポケモンや他のプレイヤーを  \n招待することも……？  \nワクワクは広がります。\n\n街でののんびり生活\n\n## ちょっぴり かわったすがたの ポケモンたち\n\nこの街で出会う、  \nちょっぴりかわった  \nすがたのポケモンを  \n覗いてみましょう。\n\nくわしくはこちら\n\n早期購入特典\n\nお店ごとにもらえる  \n早期購入特典\n\n## 商品情報\n\nタイトル\n:   『ぽこ あ ポケモン』\n\n発売日\n:   2026年3月5日（木）発売予定\n\n予約開始日\n:   【パッケージ版（キーカード）】2025年11月13日（木）  \n    【ダウンロード版】2025年11月12日（水）  \n    より順次\n\n希望小売価格\n:   【パッケージ版（キーカード）】8,980円（税込）  \n    【ダウンロード版】8,980円（税込）\n\n対応機種\n:   Nintendo Switch 2\n\nジャンル\n:   スローライフ・サンドボックス\n\nプレイ人数\n:   1人（ローカル通信・インターネット通信では最大4人）\n\n通信機能\n:   ローカル通信対応、インターネット通信対応\n\n販売形態\n:   パッケージ版（キーカード）／ダウンロード版\n\n対応言語\n:   日本語・英語・スペイン語・フランス語・ドイツ語・イタリア語・中国語（簡体字）・中国語（繁体字）・韓国語※Nintendo Switch 2 日本語・国内専用モデルでプレイされる場合も、ゲーム内で言語選択が可能です。※本ソフトの対応言語「スペイン語」は「欧州スペイン語」です。\n\nCERO\n:   A\n\n企画開発\n:   株式会社ポケモン  \n    株式会社ゲームフリーク  \n    株式会社コーエーテクモゲームス\n\n発売・販売\n:   株式会社ポケモン [...] 『ぽこ あ ポケモン』公式サイト\n\n \n\nNintendo Switch 2\n\n20